Install required dependencies

In [4]:
%pip install -r req.txt

Note: you may need to restart the kernel to use updated packages.


Import environment variables

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEYS = os.getenv('GEMINI_API_KEYS')
PROJECT_ID = os.getenv('PROJECT_ID')
DATASET = os.getenv('DATASET')
TABLE = os.getenv('TABLE')
REGION = os.getenv('REGION')

Fetch the data

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://services.google.com/fh/files/misc/startup_technical_guide_ai_agents_final.pdf")

documents = loader.load()

/home/busycaesar/projects/personal/embeddings-cosine/TorontoJS/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Split the data into chunks

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)

chunks = text_splitter.split_documents(documents)

Store the data into vector database

In [22]:
from langchain_google_community import BigQueryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

bq_vector_store = BigQueryVectorStore(
    project_id=PROJECT_ID,
    dataset_name=DATASET,
    table_name=TABLE,
    location=REGION,
    embedding=embedding_model
)

bq_vector_store.add_documents(chunks)

BigQuery table torontojs-488120.torontojs_vectordb.content initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=torontojs-488120&ws=!1m5!1m4!4m3!1storontojs-488120!2storontojs_vectordb!3scontent


['cb063f85a99041bc854cbbed2db5cfc7',
 '5476b0c11b9c40e1abb2eefed75b0c59',
 'eef2e515c02f4121b00a3eb0c4a6f3b7',
 '2c920db183d24fdc9c6b4eb215839423',
 '6744683a3da74ffebac42886e0fb5dfd',
 '2572912ff4a549fc9f43de99478d6775',
 '69a01fd7adb544939c93519c2472c8a2',
 '1999a5bd2169454e979dcdfd51eeddb1',
 '87d90a19ae9740109f41188fcf0677f4',
 'dee5f7b6c60f4e639aeddea399316a61',
 '1a26186d8c414d989ef41e510a25b17c',
 'dfdb3feee27b484086bf6ebfb180342f',
 '787c25e78b704b5e9c016bfaf220fb1a',
 '675b65f6be954b41bad98a2b50592af9',
 '6f575eb028e74f03aba8d30df130a951',
 '852a315ce4974621bae0c4a642c5a51a',
 'a5bef8fa4fe54ba4b225885f9ab10c5f',
 'd81e1740dd8c4433b89b490304218be0',
 'a882a8208a56426495a027a9f8ea5460',
 'fa8bf457c31e4b4da95065b6a31efb16',
 '33ff3f4c281143889b9a4ff96dff802c',
 '09815c58cad440a487a33af7631fa416',
 'd2a1ac3d95b94a46bb3ea3d29b4bc316',
 '6f590b1e7a0c44a8bad56dd3699bee01',
 'd45ba01294f44425a3884f6a480f1d4d',
 'c067fb1cf7064bb69379afb37d57f758',
 'c79e0fd8325144dd86cd86898b300988',
 

User's query

In [19]:
user_query = "What are the Core components for building AI agents?"

Fetch the relevant chunk of data

In [20]:
retrieved_docs = bq_vector_store.as_retriever().invoke(user_query)

retrieved_docs = " ".join([doc.page_content for doc in retrieved_docs])

Create prompt template

In [11]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["prompt", "relevant_chunk_of_data"], 
    template= 
    """
        Use the following pieces of context to answer the question at the end.

        Context: {relevant_chunk_of_data}

        User's Question: {prompt}
    """
)

Chain the prompt template with LLM and invoke it to generate the response

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI
from IPython.display import clear_output

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", google_api_key=GEMINI_API_KEYS)

chain = prompt_template | llm

response = chain.invoke({
    "prompt": user_query,
    "relevant_chunk_of_data": retrieved_docs
})

clear_output(wait=True)

print(response.content)

The core components for building AI agents are:

*   A core reasoning model
*   A set of tools to enable action
*   Data architecture options for short-term and long-term agent memory
*   A grounding mechanism to ensure factual accuracy
*   Deployment options
